# Clarté Commerce — RFM Segmentation

**Client:** Clarté Commerce S.A.S.  
**Analyst:** Nurbol Sultanov  
**Date:** February 2026  

RFM (Recency, Frequency, Monetary) segmentation of the customer base.  
Goal: compare segment distribution before and after the Q3 2023 rebranding.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

from rfm_utils import calculate_rfm, assign_rfm_labels

In [2]:
transactions = pd.read_csv('../data/raw/transactions.csv', parse_dates=['transaction_date'])
customers = pd.read_csv('../data/raw/customers.csv', parse_dates=['registration_date'])

print(f'Transactions: {transactions.shape}')
print(f'Customers: {customers.shape}')

Transactions: (92330, 11)
Customers: (12005, 9)


In [3]:
# Filter out test orders
mask = transactions['customer_id'].str.contains('TEST')
print(f'Filtered out {mask.sum()} test transactions')
transactions = transactions[~mask]
print(f'Working with {len(transactions):,} transactions')

Filtered out 20 test transactions
Working with 92,310 transactions


## 1. Calculate RFM Scores

In [4]:
snapshot_date = pd.Timestamp('2024-12-31')

rfm = calculate_rfm(transactions, snapshot_date)

print(f'RFM table shape: {rfm.shape}')
print()
print(rfm[['recency', 'frequency', 'monetary']].describe().round(3).to_string())

RFM table shape: (11572, 7)

         recency  frequency    monetary
count  11572.000  11572.000   11572.000
mean     741.223      7.978     886.432
std      318.456      5.621     724.891
min       31.000      1.000       4.530
25%      482.000      3.000     327.150
50%      731.000      6.000     672.800
75%      987.000     11.000    1218.640
max     1461.000     42.000    6842.100


## 2. RFM Score Distribution

In [5]:
# Assign quintile scores
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1])
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])

rfm['r_score'] = rfm['r_score'].astype(int)
rfm['f_score'] = rfm['f_score'].astype(int)
rfm['m_score'] = rfm['m_score'].astype(int)
rfm['rfm_total'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col, color in zip(axes, ['r_score', 'f_score', 'm_score'], 
                           ['#e74c3c', '#3498db', '#27ae60']):
    rfm[col].value_counts().sort_index().plot(kind='bar', ax=ax, color=color, edgecolor='white')
    ax.set_title(f'{col.upper()} Distribution')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)

plt.suptitle('RFM Score Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Segment Labels

In [6]:
rfm = assign_rfm_labels(rfm)

segment_counts = rfm['rfm_label'].value_counts()
segment_pcts = rfm['rfm_label'].value_counts(normalize=True)

print('Segment distribution:\n')
for label in segment_counts.index:
    print(f'{label:22s} {segment_counts[label]:5d}  {segment_pcts[label]:.1%}')

Segment distribution:

rfm_label
Other                 3841  33.2%
Champions             1648  14.2%
Lost                  1587  13.7%
Loyal Customers       1432  12.4%
At Risk               1203  10.4%
Potential Loyalists    1024   8.8%
New Customers          512   4.4%
Cant Lose Them         325   2.8%


In [7]:
# Segment distribution pie chart
colors = {
    'Champions': '#27ae60',
    'Loyal Customers': '#2ecc71',
    'Potential Loyalists': '#3498db',
    'New Customers': '#1abc9c',
    'At Risk': '#e67e22',
    'Cant Lose Them': '#e74c3c',
    'Lost': '#95a5a6',
    'Other': '#bdc3c7'
}

fig, ax = plt.subplots(figsize=(10, 8))
segment_counts.plot(
    kind='pie', ax=ax, autopct='%1.1f%%',
    colors=[colors.get(x, '#bdc3c7') for x in segment_counts.index],
    startangle=90, pctdistance=0.85
)
ax.set_ylabel('')
ax.set_title('Customer Segments (RFM)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/rfm_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Segment Profiles

In [8]:
segment_profiles = rfm.groupby('rfm_label').agg(
    count=('customer_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    avg_rfm_total=('rfm_total', 'mean')
).round(1)

segment_profiles = segment_profiles.sort_values('avg_rfm_total', ascending=False)

print('Average metrics by segment:')
print(segment_profiles.to_string())

Average metrics by segment:


## 5. RFM Heatmap

In [9]:
rfm_heatmap = rfm.groupby(['r_score', 'f_score']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(rfm_heatmap, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Customer Count by R and F Scores', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequency Score')
ax.set_ylabel('Recency Score')

plt.tight_layout()
plt.show()

## 6. Export

Exporting RFM scores for dashboard and further analysis.

In [10]:
rfm.to_csv('../data/processed/rfm_scores.csv', index=False)
print(f'Exported rfm_scores.csv ({len(rfm)} rows)')

Exported rfm_scores.csv (11572 rows)


## Notes

- RFM calculated using snapshot date 2024-12-31
- Quintile-based scoring (1-5 for each dimension)
- Next: compare segment distribution pre vs post rebrand
- Next: overlay with churn analysis